Torch vs pywavlet wavelet transform comparison

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import pywt
import time
import torch.nn as nn

#wavlet decomposition into 2 levels 

def wavelet_decomposition_level2(image,wavelet = 'db4'):
    """
        Performs 3-level wavelet demoposition
        Returns
            dict with:
            level1: (LL1, LH1, HL1, HH1)
            level2: (LL2, LH2, HL2, HH2)
    """
    LL1, (LH1, HL1, HH1) = pywt.dwt2(image, wavelet, mode='reflect')

    # ---------------- LEVEL 2 ----------------
    LL2, (LH2, HL2, HH2) = pywt.dwt2(LL1, wavelet, mode='reflect')

    # ---------------- LEVEL 3 ----------------

    return {
        "level1": (LL1, LH1, HL1, HH1),
        "level2": (LL2, LH2, HL2, HH2),
    }




In [ ]:


# =========================================================
# Basic CNN block (you can replace freely)
# =========================================================
class StreamCNN(nn.Module):
    def __init__(self, in_ch=1, feat_dim=32):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 16, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(1)
        )

    def forward(self, x):
        return self.net(x).view(x.size(0), -1)


# =========================================================
# Main Wavelet Pyramid Model
# =========================================================
class WaveletPyramidNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        # one network per frequency band
        self.ll_net = StreamCNN()
        self.lh_net = StreamCNN()
        self.hl_net = StreamCNN()
        self.hh_net = StreamCNN()

        # fusion + classifier
        self.classifier = nn.Sequential(
            nn.Linear(32 * 4, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    # ---------------------------------------------------------
    # You will fill this part yourself (important abstraction)
    # ---------------------------------------------------------
    def process_levels(self, net, levels):
        """
        levels: list of tensors [L1, L2, L3]

        EXPECTATION:
        - each level already is a tensor (C, H, W) or (B, C, H, W)
        - you decide preprocessing / resizing / normalization
        """

        features = []

        for x in levels:
            # TODO: you decide preprocessing here
            # example: x = x.unsqueeze(0) or resize, normalize, etc.

            f = net(x)
            features.append(f)

        return torch.mean(torch.stack(features, dim=0), dim=0)

    # ---------------------------------------------------------
    # forward expects precomputed wavelet tensors
    # ---------------------------------------------------------
    def forward(self, LL_levels, LH_levels, HL_levels, HH_levels):

        ll = self.process_levels(self.ll_net, LL_levels)
        lh = self.process_levels(self.lh_net, LH_levels)
        hl = self.process_levels(self.hl_net, HL_levels)
        hh = self.process_levels(self.hh_net, HH_levels)

        # fusion
        x = torch.cat([ll, lh, hl, hh], dim=1)

        return self.classifier(x)


In [ ]:
import torch
import torch.nn as nn

class WaveletHybridNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        # =====================================================
        # BASIC BLOCKS
        # =====================================================
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        # =====================================================
        # RGB BRANCH
        # Input: (B, 3, H, W)
        # Output: (B, 128, H/4, W/4)
        # =====================================================

        self.rgb_conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.rgb_bn1   = nn.BatchNorm2d(32)

        self.rgb_conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.rgb_bn2   = nn.BatchNorm2d(64)

        self.rgb_conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.rgb_bn3   = nn.BatchNorm2d(128)

        # Refinement (keeps same spatial size)
        self.rgb_refine1 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.rgb_refine2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)


        # =====================================================
        # WAVELET BRANCH
        # Each level input: (B, 12, ...)
        # (LL, LH, HL, HH) each 3 channels → 12 total
        # =====================================================

        # ---- Level 1 (H/2 → pooled to H/4) ----
        self.wav_conv_l1_1 = nn.Conv2d(12, 64, kernel_size=3, padding=1)
        self.wav_conv_l1_2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        # ---- Level 2 (H/4) ----
        self.wav_conv_l2_1 = nn.Conv2d(12, 64, kernel_size=3, padding=1)
        self.wav_conv_l2_2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        # ---- Wavelet refinement ----
        self.wav_refine1 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.wav_refine2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)

        # ---- Projection for residual fusion ----
        self.wav_proj = nn.Conv2d(128, 128, kernel_size=1)


        # =====================================================
        # LEVEL ATTENTION (L1 vs L2)
        # Input: (B, 256, H/4, W/4)
        # Output: (B, 2)
        # =====================================================
        self.level_attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),   # (B, 256, 1, 1)
            nn.Flatten(),              # (B, 256)
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
            nn.Softmax(dim=1)
        )


        # =====================================================
        # FINAL ATTENTION (RGB vs Wavelet)
        # Input: concat(rgb_vec, wav_vec) → (B, 256)
        # Output: (B, 2)
        # =====================================================
        self.final_attn = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
            nn.Softmax(dim=1)
        )


        # =====================================================
        # CLASSIFIER
        # Input: (B, 128)
        # =====================================================
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )


In [ ]:
import torch
import torch.nn as nn


# =========================================================
# MODEL
# =========================================================
class WaveletHybridNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        # ---------------- RGB ----------------
        self.rgb_conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.rgb_bn1   = nn.BatchNorm2d(32)

        self.rgb_conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.rgb_bn2   = nn.BatchNorm2d(64)

        self.rgb_conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.rgb_bn3   = nn.BatchNorm2d(128)

        self.rgb_refine1 = nn.Conv2d(128, 128, 3, padding=1)
        self.rgb_refine2 = nn.Conv2d(128, 128, 3, padding=1)

        # ---------------- WAVELET ----------------
        self.wav_conv_l1_1 = nn.Conv2d(12, 64, 3, padding=1)
        self.wav_conv_l1_2 = nn.Conv2d(64, 128, 3, padding=1)

        self.wav_conv_l2_1 = nn.Conv2d(12, 64, 3, padding=1)
        self.wav_conv_l2_2 = nn.Conv2d(64, 128, 3, padding=1)

        self.wav_refine1 = nn.Conv2d(128, 128, 3, padding=1)
        self.wav_refine2 = nn.Conv2d(128, 128, 3, padding=1)

        self.wav_proj = nn.Conv2d(128, 128, 1)

        # ---------------- ATTENTION ----------------
        self.level_attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
            nn.Softmax(dim=1)
        )

        self.final_attn = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
            nn.Softmax(dim=1)
        )

        # ---------------- CLASSIFIER ----------------
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    # =====================================================
    # FORWARD
    # =====================================================
    def forward(self, x_rgb, wav):
        B = x_rgb.size(0)

        # ---------------- RGB ----------------
        rgb = self.pool(self.relu(self.rgb_bn1(self.rgb_conv1(x_rgb))))
        rgb = self.pool(self.relu(self.rgb_bn2(self.rgb_conv2(rgb))))
        rgb = self.relu(self.rgb_bn3(self.rgb_conv3(rgb)))

        # ---------------- WAVELET LEVEL 1 ----------------
        L1 = torch.cat([wav["LL1"], wav["LH1"], wav["HL1"], wav["HH1"]], dim=1)
        L1 = self.relu(self.wav_conv_l1_1(L1))
        L1 = self.relu(self.wav_conv_l1_2(L1))
        L1 = self.pool(L1)

        # ---------------- WAVELET LEVEL 2 ----------------
        L2 = torch.cat([wav["LL2"], wav["LH2"], wav["HL2"], wav["HH2"]], dim=1)
        L2 = self.relu(self.wav_conv_l2_1(L2))
        L2 = self.relu(self.wav_conv_l2_2(L2))

        # ---------------- LEVEL ATTENTION ----------------
        L_cat = torch.cat([L1, L2], dim=1)  # (B, 256, H/4, W/4)

        w = self.level_attn(L_cat)  # (B, 2)
        w1 = w[:, 0].view(B, 1, 1, 1)
        w2 = w[:, 1].view(B, 1, 1, 1)

        W = w1 * L1 + w2 * L2

        # ---------------- RESIDUAL FUSION ----------------
        W_proj = self.wav_proj(W)
        rgb = rgb + W_proj

        # ---------------- REFINEMENT ----------------
        rgb = self.relu(self.rgb_refine1(rgb))
        rgb = self.relu(self.rgb_refine2(rgb))

        W = self.relu(self.wav_refine1(W))
        W = self.relu(self.wav_refine2(W))

        # ---------------- GLOBAL POOL ----------------
        rgb_vec = self.global_pool(rgb).view(B, -1)
        wav_vec = self.global_pool(W).view(B, -1)

        # ---------------- FINAL ATTENTION ----------------
        f = torch.cat([rgb_vec, wav_vec], dim=1)

        w_final = self.final_attn(f)
        f = w_final[:, 0:1] * rgb_vec + w_final[:, 1:2] * wav_vec

        # ---------------- CLASSIFIER ----------------
        return self.classifier(f)


# =========================================================
# DUMMY DATA GENERATION
# =========================================================
def create_dummy_data(B=2, H=128, W=128):
    x_rgb = torch.randn(B, 3, H, W)

    wav = {
        "LL1": torch.randn(B, 3, H//2, W//2),
        "LH1": torch.randn(B, 3, H//2, W//2),
        "HL1": torch.randn(B, 3, H//2, W//2),
        "HH1": torch.randn(B, 3, H//2, W//2),

        "LL2": torch.randn(B, 3, H//4, W//4),
        "LH2": torch.randn(B, 3, H//4, W//4),
        "HL2": torch.randn(B, 3, H//4, W//4),
        "HH2": torch.randn(B, 3, H//4, W//4),
    }

    return x_rgb, wav


# =========================================================
# FORWARD TEST
# =========================================================
if __name__ == "__main__":

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = WaveletHybridNet().to(device)
    model.eval()

    x_rgb, wav = create_dummy_data()

    x_rgb = x_rgb.to(device)
    wav = {k: v.to(device) for k, v in wav.items()}

    with torch.no_grad():
        out = model(x_rgb, wav)

    print("\n===== FORWARD TEST =====")
    print("Output shape:", out.shape)
    print("Output logits:", out)
